# CompoCLIP role-3 GPU evaluation (Colab + VS Code)

Upload `A2_eval.tar`, `A2_models.tar`, `data_role3.tar` to `MyDrive/compoclip/` first. Then select the Colab T4 kernel and run all cells.

See `docs/colab-vscode.md` for the full explanation.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/wh-violet667/CompoCLIP.git
%cd CompoCLIP

In [ ]:
!mkdir -p /content/compoclip_data
!tar -xf /content/drive/MyDrive/compoclip/A2_eval.tar -C /content/compoclip_data
!tar -xf /content/drive/MyDrive/compoclip/A2_models.tar -C /content/compoclip_data
!tar -xf /content/drive/MyDrive/compoclip/data_role3.tar -C /content/compoclip_data

In [ ]:
!ln -sfn /content/compoclip_data/A2_eval /content/CompoCLIP/A2_eval
!ln -sfn /content/compoclip_data/A2_models /content/CompoCLIP/A2_models
!ln -sfn /content/compoclip_data/data/golden /content/CompoCLIP/data/golden
!ln -sfn /content/compoclip_data/data/flickr30k /content/CompoCLIP/data/flickr30k
!ln -sfn /content/compoclip_data/data/winoground /content/CompoCLIP/data/winoground

In [ ]:
!pip -q install "open_clip_torch==2.24.0" timm ftfy regex huggingface_hub safetensors spacy pandas tqdm pyyaml datasets faiss-cpu
!python -m spacy download en_core_web_sm

In [ ]:
!python scripts/make_a2_eval_portable.py \
  --root /content/CompoCLIP/A2_eval \
  --data-root /content/CompoCLIP/data

In [ ]:
import json
from pathlib import Path
p = Path('/content/CompoCLIP/data/winoground/pairs.jsonl')
rows = [json.loads(line) for line in p.read_text(encoding='utf-8').splitlines() if line.strip()]
img_root = Path('/content/CompoCLIP/data/winoground/images')
for r in rows:
    r['image_0'] = str(img_root / Path(str(r['image_0']).replace('\\', '/')).name)
    r['image_1'] = str(img_root / Path(str(r['image_1']).replace('\\', '/')).name)
p.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + '\n', encoding='utf-8')
print('winoground paths fixed', len(rows))

In [ ]:
!python scripts/evaluate_role3_models.py \
  --tags e3 e4b \
  --benchmarks aro sugarcrepe coco \
  --device cuda --image-batch 64 --text-batch 256

In [ ]:
!for tag in e1 e2 e2a e3 e4b; do \
  python evals-mm/run_eval.py --tag "$tag" \
    --checkpoint "A2_models/$tag/${tag}_model.pt" --benchmark flickr30k; \
  python evals-mm/run_eval.py --tag "$tag" \
    --checkpoint "A2_models/$tag/${tag}_model.pt" --benchmark winoground; \
done

In [ ]:
!python scripts/assemble_role3_results.py
!python scripts/build_full_master.py
!python scripts/make_master_md.py

In [ ]:
!mkdir -p /content/drive/MyDrive/compoclip/results_role3
!cp -r report/results/* /content/drive/MyDrive/compoclip/results_role3/
!cp -r A2_eval/results/e3 /content/drive/MyDrive/compoclip/results_role3/
!cp -r A2_eval/results/e4b /content/drive/MyDrive/compoclip/results_role3/
!cp -r evals-mm/results/*.csv /content/drive/MyDrive/compoclip/results_role3/

## Notes

- `/content` resets when the runtime disconnects; re-run the extraction cells.
- Copy results to Drive after each long job.
- `assemble_role3_results.py` updates the common 2/4 ARO + 2/3 Sugar table. The Flickr/Winoground/COCO rows are in `evals-mm/results/*.csv` and `A2_eval/results/*/evaluation.json`; fill the remaining report cells from those files.
- Do not use scripted keep-alive; it violates Colab terms.